In [ ]:
!pip install torchvision
!pip install fvcore umap-learn scikit-video opencv-python-headless matplotlib seaborn tqdm scikit-learn imblearn albumentations

In [ ]:
# Install required dependencies
!pip install fvcore umap-learn

import os
import cv2
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.parallel  # For DataParallel
import torch.hub  # Use torch.hub for model loading
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.preprocessing import LabelEncoder, StandardScaler, RobustScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    confusion_matrix, roc_curve, auc, balanced_accuracy_score,
    matthews_corrcoef, silhouette_score
)
from sklearn.feature_selection import SelectKBest, f_classif
from scipy.stats import kendalltau, spearmanr, entropy, wasserstein_distance
from scipy.linalg import sqrtm
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
from collections import Counter
import logging
import warnings
from imblearn.over_sampling import SMOTE
import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Enhanced Configuration (aligned with first code)
CONFIG = {
    'input_folder': "/kaggle/input/tvsum-dataset/tvsum_dataset/ydata-tvsum50-video/video",
    'output_folder': "/kaggle/working",
    'annotation_file': "/kaggle/input/tvsum-dataset/tvsum_dataset/ydata-tvsum50-data/data/ydata-tvsum50-anno.tsv",
    'seed': 42,
    'test_size': 0.2,
    'val_size': 0.1,
    'min_samples_per_class': 3,
    'max_samples_per_class': 50
}

# Set random seeds for reproducibility
def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seeds(CONFIG['seed'])

# Enhanced annotation loading with data validation (from first code)
def load_annotation_file(annotation_path: str) -> dict:
    try:
        df = pd.read_csv(annotation_path, sep='\t', header=None, encoding='utf-8-sig')
        annotations = {}
        category_counts = Counter()
        
        for _, row in df.iterrows():
            video_name = row[0].strip()
            category = row[1].strip() if pd.notna(row[1]) else 'Unknown'
            annotations[video_name] = {'category': category}
            category_counts[category] += 1
        
        valid_categories = {cat for cat, count in category_counts.items() if count >= CONFIG['min_samples_per_class']}
        annotations = {k: v for k, v in annotations.items() if v['category'] in valid_categories}
        
        logging.info(f"Loaded annotations for {len(annotations)} videos")
        logging.info(f"Valid categories: {valid_categories}")
        logging.info(f"Category distribution: {dict(category_counts)}")
        
        return annotations
    except Exception as e:
        logging.error(f"Failed to load annotation file: {e}")
        return {}

# Helper functions for video paths and labels (from first code)
def get_video_paths_with_annotations(directory: str, annotations: dict) -> list:
    video_extensions = ['.mp4', '.avi', '.mov', '.mkv']
    video_paths = []
    
    if not os.path.exists(directory):
        logging.error(f"Directory does not exist: {directory}")
        return []
    
    video_files = [f for f in os.listdir(directory) 
                   if any(f.lower().endswith(ext) for ext in video_extensions)]
    
    for file in video_files:
        video_name = os.path.splitext(file)[0]
        if video_name in annotations or file in annotations:
            video_paths.append(os.path.join(directory, file))
        else:
            for ann_key in annotations.keys():
                if video_name.lower() == ann_key.lower() or file.lower() == ann_key.lower():
                    video_paths.append(os.path.join(directory, file))
                    break
    
    logging.info(f"Matched {len(video_paths)} videos with annotations out of {len(video_files)} total videos")
    return video_paths

def get_annotation_labels(video_paths: list, annotations: dict) -> list:
    labels = []
    missing_annotations = []
    
    for video_path in video_paths:
        filename = os.path.basename(video_path)
        video_name = os.path.splitext(filename)[0]
        category = None
        
        if video_name in annotations:
            category = annotations[video_name]['category']
        elif filename in annotations:
            category = annotations[filename]['category']
        else:
            for ann_key in annotations.keys():
                if video_name.lower() == ann_key.lower() or filename.lower() == ann_key.lower():
                    category = annotations[ann_key]['category']
                    break
        
        if category is None:
            category = 'Unknown'
            missing_annotations.append(filename)
        
        labels.append(category)
    
    if missing_annotations:
        logging.warning(f"No category found for {len(missing_annotations)} videos: {missing_annotations[:5]}...")
    
    return labels

# Enhanced VideoDataset with optical flow variance (from first code)
class EnhancedVideoDataset(Dataset):
    def __init__(self, video_paths, labels, label_encoder, mode='train', transform=None):
        self.video_paths = video_paths
        self.labels = labels
        self.label_encoder = label_encoder
        self.mode = mode
        self.transform = transform
        
        if mode == 'train':
            self.video_augmentation = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
                A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
                A.GaussianBlur(blur_limit=(3, 7), p=0.2),
                A.CoarseDropout(max_holes=8, max_height=4, max_width=8, p=0.3),
                A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ToTensorV2()
            ])
        else:
            self.video_augmentation = A.Compose([
                A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ToTensorV2()
            ])
        
        self.temporal_features = self._compute_temporal_features()
        self.flow_variances = self._compute_all_flow_variances()

    def _compute_temporal_features(self):
        features = []
        for video_path in tqdm(self.video_paths, desc=f"Computing temporal features ({self.mode})"):
            feature = self._extract_temporal_features(video_path)
            features.append(feature)
        return np.array(features)
    
    def _extract_temporal_features(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        
        n_samples = min(32, total_frames)
        indices = np.linspace(0, total_frames - 1, n_samples, dtype=int)
        
        frames = []
        gray_frames = []
        
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                frame_resized = cv2.resize(frame, (224, 224))
                frames.append(frame_resized)
                gray_frames.append(cv2.cvtColor(frame_resized, cv2.COLOR_BGR2GRAY))
        
        cap.release()
        
        if len(frames) < 2:
            return np.zeros(10)
        
        features = []
        
        flows = []
        for i in range(len(gray_frames) - 1):
            flow = cv2.calcOpticalFlowFarneback(
                gray_frames[i], gray_frames[i + 1], None, 
                0.5, 3, 15, 3, 5, 1.2, 0
            )
            mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
            flows.append(mag)
        
        if flows:
            flow_stats = [
                np.mean([np.mean(f) for f in flows]),
                np.mean([np.std(f) for f in flows]),
                np.mean([np.max(f) for f in flows]),
                np.std([np.mean(f) for f in flows])
            ]
        else:
            flow_stats = [0, 0, 0, 0]
        
        features.extend(flow_stats)
        
        frame_diffs = []
        for i in range(len(frames) - 1):
            diff = cv2.absdiff(frames[i], frames[i + 1])
            frame_diffs.append(np.mean(diff))
        
        if frame_diffs:
            diff_stats = [
                np.mean(frame_diffs),
                np.std(frame_diffs),
                np.max(frame_diffs)
            ]
        else:
            diff_stats = [0, 0, 0]
        
        features.extend(diff_stats)
        
        features.extend([
            total_frames,
            fps,
            total_frames / max(fps, 1)
        ])
        
        return np.array(features)

    def _compute_flow_variance(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        indices = np.linspace(0, total_frames - 1, 16, dtype=int)
        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                frames.append(frame)
        cap.release()
        flows = []
        for i in range(len(frames) - 1):
            flow = cv2.calcOpticalFlowFarneback(frames[i], frames[i + 1], None, 0.5, 3, 15, 3, 5, 1.2, 0)
            mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
            flows.append(mag)
        if not flows:
            return 0
        return np.mean([np.var(f) for f in flows])

    def _compute_all_flow_variances(self):
        variances = []
        for video_path in tqdm(self.video_paths, desc="Computing Flow Variance"):
            variances.append(self._compute_flow_variance(video_path))
        return np.array(variances)

    def __len__(self):
        return len(self.video_paths)
    
    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        class_name = self.labels[idx]
        
        try:
            frames = self._extract_frames_robust(video_path)
            label = self.label_encoder.transform([class_name])[0]
            temporal_feat = self.temporal_features[idx]
            
            return frames, label, temporal_feat
        except Exception as e:
            logging.warning(f"Error processing video {video_path}: {e}")
            dummy_frames = torch.zeros(32, 3, 224, 224)
            dummy_label = 0
            dummy_temporal = np.zeros(10)
            return dummy_frames, dummy_label, dummy_temporal
    
    def _extract_frames_robust(self, video_path):
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise ValueError(f"Could not open video: {video_path}")
        
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames == 0:
            raise ValueError(f"Video has no frames: {video_path}")
        
        indices = np.linspace(0, total_frames - 1, 32, dtype=int)
        frames = []
        
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret and frame is not None:
                frame = cv2.resize(frame, (224, 224))
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                
                if self.mode == 'train':
                    augmented = self.video_augmentation(image=frame)
                    frame_tensor = augmented['image']
                else:
                    normalized = self.video_augmentation(image=frame)
                    frame_tensor = normalized['image']
                
                frames.append(frame_tensor)
        
        cap.release()
        
        if len(frames) < 32:
            last_frame = frames[-1] if frames else torch.zeros(3, 224, 224)
            frames.extend([last_frame.clone() for _ in range(32 - len(frames))])
        elif len(frames) > 32:
            frames = frames[:32]
        
        return torch.stack(frames).permute(1, 0, 2, 3)

# Define the VideoFeatureExtractor class using X3D (kept from second code)
class VideoFeatureExtractor(nn.Module):
    def __init__(self, pretrained=True):
        super(VideoFeatureExtractor, self).__init__()
        # Load X3D-M model from PyTorchVideo via torch.hub
        self.model = torch.hub.load('facebookresearch/pytorchvideo', 'x3d_m', pretrained=pretrained)
        # Remove the classification head to get features
        self.model.blocks[-1].proj = nn.Identity()  # Replace the projection head with Identity

    def forward(self, x):
        # Input x: (batch_size, C, T, H, W), e.g., (batch_size, 3, 32, 224, 224)
        features = self.model(x)  # Pass x directly to the model
        if features.dim() > 2:
            features = torch.mean(features, dim=[2, 3, 4])  # Global average pooling if needed
        return features  # Output: (batch_size, feature_dim)

# Enhanced Hybrid Classifier (from first code)
class RobustHybridClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_neighbors=7, C=1.0, gamma='scale', kernel='rbf', 
                 svm_weight=0.4, knn_weight=0.3, rf_weight=0.3, n_estimators=100):
        self.n_neighbors = n_neighbors
        self.C = C
        self.gamma = gamma
        self.kernel = kernel
        self.svm_weight = svm_weight
        self.knn_weight = knn_weight
        self.rf_weight = rf_weight
        self.n_estimators = n_estimators
        self.knn = None
        self.svm = None
        self.rf = None
        self.classes_ = None
    
    def fit(self, X, y):
        # Cap n_neighbors to the number of samples
        self.n_neighbors = min(self.n_neighbors, X.shape[0])
        self.knn = KNeighborsClassifier(
            n_neighbors=self.n_neighbors,
            weights='distance',
            metric='cosine'
        ).fit(X, y)
        
        self.svm = SVC(
            C=self.C,
            gamma=self.gamma,
            kernel=self.kernel,
            probability=True,
            class_weight='balanced'
        ).fit(X, y)
        
        self.rf = RandomForestClassifier(
            n_estimators=self.n_estimators,
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=2,
            class_weight='balanced',
            random_state=42
        ).fit(X, y)
        
        self.classes_ = self.svm.classes_
        return self
    
    def predict_proba(self, X):
        knn_probs = self._get_knn_proba(X)
        svm_probs = self.svm.predict_proba(X)
        rf_probs = self.rf.predict_proba(X)
        ensemble_probs = (self.knn_weight * knn_probs + 
                         self.svm_weight * svm_probs + 
                         self.rf_weight * rf_probs)
        return ensemble_probs
    
    def predict(self, X):
        probs = self.predict_proba(X)
        return np.argmax(probs, axis=1)
    
    def _get_knn_proba(self, X):
        distances, indices = self.knn.kneighbors(X)
        neighbor_labels = self.knn._y[indices]
        weights = 1.0 / (distances + 1e-8)
        probas = np.zeros((X.shape[0], len(self.classes_)))
        for i in range(X.shape[0]):
            for j, label in enumerate(neighbor_labels[i]):
                probas[i, label] += weights[i, j]
            if probas[i].sum() > 0:
                probas[i] /= probas[i].sum()
        return probas

# Enhanced feature extraction with temporal features (from first code)
def extract_enhanced_features(model, loader, device='cuda'):
    model.to(device)
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model)  # Wrap model for multi-GPU support
    
    model.eval()
    visual_features, temporal_features, labels = [], [], []
    
    with torch.no_grad():
        for frames, lbls, temporal_feats in tqdm(loader, desc="Extracting Enhanced Features"):
            frames = frames.to(device)  # Shape: (batch_size, C, T, H, W)
            vis_feats = model(frames)   # Shape: (batch_size, feature_dim)
            visual_features.append(vis_feats.cpu().numpy())
            temporal_features.append(temporal_feats.numpy())
            labels.extend(lbls.numpy())
    
    visual_features = np.vstack(visual_features)
    temporal_features = np.vstack(temporal_features)
    combined_features = np.hstack([visual_features, temporal_features])
    
    return combined_features, np.array(labels)

# Fréchet Video Distance (aligned with first code)
def calculate_fvd(real_features, fake_features):
    mu1, sigma1 = np.mean(real_features, axis=0), np.cov(real_features, rowvar=False)
    mu2, sigma2 = np.mean(fake_features, axis=0), np.cov(fake_features, rowvar=False)
    diff = np.sum((mu1 - mu2)**2)
    covmean = sqrtm(sigma1.dot(sigma2))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff + np.trace(sigma1 + sigma2 - 2 * covmean)
    return fid

# Enhanced evaluation (from first code)
def enhanced_evaluate(y_true, y_pred, y_proba, label_encoder, train_features, test_features, 
                     train_temporal, test_temporal, train_flow_variances, test_flow_variances, fold_id=None):
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted'),
        'f1_macro': f1_score(y_true, y_pred, average='macro'),
        'precision_weighted': precision_score(y_true, y_pred, average='weighted'),
        'recall_weighted': recall_score(y_true, y_pred, average='weighted'),
        'mcc': matthews_corrcoef(y_true, y_pred),
        'kendalltau': kendalltau(y_true, y_pred)[0] if len(np.unique(y_true)) > 1 else 0,
        'spearmanr': spearmanr(y_true, y_pred)[0] if len(np.unique(y_true)) > 1 else 0
    }
    
    n_unique_labels = len(np.unique(y_true))
    if n_unique_labels > 1 and len(y_true) > n_unique_labels:
        try:
            metrics['silhouette'] = silhouette_score(test_features, y_true)
        except:
            metrics['silhouette'] = 0.0
    else:
        metrics['silhouette'] = 0.0
    
    if fold_id is not None:
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=label_encoder.classes_,
                    yticklabels=label_encoder.classes_)
        plt.title(f'Confusion Matrix - Fold {fold_id}')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.tight_layout()
        plt.savefig(f"{CONFIG['output_folder']}/confusion_matrix_fold_{fold_id}.png", dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        
        # Define per-class metrics before plotting
        per_class_f1 = f1_score(y_true, y_pred, average=None)
        per_class_precision = precision_score(y_true, y_pred, average=None)
        per_class_recall = recall_score(y_true, y_pred, average=None)
        
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 3, 1)
        sns.barplot(x=label_encoder.classes_, y=per_class_f1, palette='Blues')
        plt.title('Per-Class F1 Score')
        plt.xticks(rotation=45)
        plt.ylabel('F1 Score')
        plt.subplot(1, 3, 2)
        sns.barplot(x=label_encoder.classes_, y=per_class_precision, palette='Greens')
        plt.title('Per-Class Precision')
        plt.xticks(rotation=45)
        plt.ylabel('Precision')
        plt.subplot(1, 3, 3)
        sns.barplot(x=label_encoder.classes_, y=per_class_recall, palette='Reds')
        plt.title('Per-Class Recall')
        plt.xticks(rotation=45)
        plt.ylabel('Recall')
        plt.tight_layout()
        plt.savefig(f"{CONFIG['output_folder']}/per_class_metrics_fold_{fold_id}.png", dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        
        if train_features.shape[0] + test_features.shape[0] > 50:
            combined_features = np.vstack([train_features, test_features])
            combined_labels = np.hstack([
                ['Train'] * len(train_features),
                ['Test'] * len(test_features)
            ])
            if combined_features.shape[1] > 50:
                pca = PCA(n_components=50, random_state=42)
                combined_features_reduced = pca.fit_transform(combined_features)
            else:
                combined_features_reduced = combined_features
            
            try:
                tsne = TSNE(n_components=2, random_state=42, 
                           perplexity=min(30, len(combined_features_reduced)//4))
                tsne_result = tsne.fit_transform(combined_features_reduced)
                
                plt.figure(figsize=(12, 5))
                plt.subplot(1, 2, 1)
                scatter = plt.scatter(tsne_result[:, 0], tsne_result[:, 1], 
                                    c=[0 if x == 'Train' else 1 for x in combined_labels],
                                    cmap='RdYlBu', alpha=0.7)
                plt.title('t-SNE: Train vs Test Distribution')
                plt.colorbar(scatter, ticks=[0, 1], label='Train (0) / Test (1)')
                plt.subplot(1, 2, 2)
                all_true_labels = np.hstack([
                    train_labels[:len(train_features)],
                    y_true
                ])
                scatter2 = plt.scatter(tsne_result[:, 0], tsne_result[:, 1], 
                                     c=all_true_labels, cmap='tab10', alpha=0.7)
                plt.title('t-SNE: Class Distribution')
                plt.colorbar(scatter2)
                plt.tight_layout()
                plt.savefig(f"{CONFIG['output_folder']}/tsne_fold_{fold_id}.png", dpi=300, bbox_inches='tight')
                plt.show()
                plt.close()
            except Exception as e:
                logging.warning(f"Could not create t-SNE visualization: {e}")
        
        # Temporal feature visualization (t-SNE and UMAP)
        combined_temporal = np.vstack([train_temporal, test_temporal])
        combined_labels = np.hstack([
            ['Train'] * len(train_temporal),
            ['Test'] * len(test_temporal)
        ])
        all_true_labels = np.hstack([
            train_labels[:len(train_temporal)],
            y_true
        ])
        
        # t-SNE for temporal features
        try:
            tsne_temporal = TSNE(n_components=2, random_state=42, 
                                perplexity=min(30, len(combined_temporal)//4))
            tsne_temporal_result = tsne_temporal.fit_transform(combined_temporal)
            
            plt.figure(figsize=(12, 5))
            plt.subplot(1, 2, 1)
            scatter = plt.scatter(tsne_temporal_result[:, 0], tsne_temporal_result[:, 1], 
                                c=[0 if x == 'Train' else 1 for x in combined_labels],
                                cmap='RdYlBu', alpha=0.7)
            plt.title(f't-SNE Temporal: Train vs Test (Fold {fold_id})')
            plt.colorbar(scatter, ticks=[0, 1], label='Train (0) / Test (1)')
            plt.subplot(1, 2, 2)
            scatter2 = plt.scatter(tsne_temporal_result[:, 0], tsne_temporal_result[:, 1], 
                                 c=all_true_labels, cmap='tab10', alpha=0.7)
            plt.title(f't-SNE Temporal: Class Distribution (Fold {fold_id})')
            plt.colorbar(scatter2)
            plt.tight_layout()
            plt.savefig(f"{CONFIG['output_folder']}/tsne_temporal_fold_{fold_id}.png", dpi=300, bbox_inches='tight')
            plt.show()
            plt.close()
        except Exception as e:
            logging.warning(f"Could not create t-SNE temporal visualization: {e}")
        
        # UMAP for temporal features
        try:
            umap_reducer = umap.UMAP(n_components=2, random_state=42, 
                                    n_neighbors=min(15, len(combined_temporal)//2))
            umap_result = umap_reducer.fit_transform(combined_temporal)
            
            plt.figure(figsize=(12, 5))
            plt.subplot(1, 2, 1)
            scatter = plt.scatter(umap_result[:, 0], umap_result[:, 1], 
                                c=[0 if x == 'Train' else 1 for x in combined_labels],
                                cmap='RdYlBu', alpha=0.7)
            plt.title(f'UMAP Temporal: Train vs Test (Fold {fold_id})')
            plt.colorbar(scatter, ticks=[0, 1], label='Train (0) / Test (1)')
            plt.subplot(1, 2, 2)
            scatter2 = plt.scatter(umap_result[:, 0], umap_result[:, 1], 
                                 c=all_true_labels, cmap='tab10', alpha=0.7)
            plt.title(f'UMAP Temporal: Class Distribution (Fold {fold_id})')
            plt.colorbar(scatter2)
            plt.tight_layout()
            plt.savefig(f"{CONFIG['output_folder']}/umap_temporal_fold_{fold_id}.png", dpi=300, bbox_inches='tight')
            plt.show()
            plt.close()
        except Exception as e:
            logging.warning(f"Could not create UMAP temporal visualization: {e}")
        
        # Optical Flow Variance Visualization
        plt.figure(figsize=(10, 6))
        sns.histplot(train_flow_variances, kde=True, label='Train', alpha=0.5, color='blue')
        sns.histplot(test_flow_variances, kde=True, label='Test', alpha=0.5, color='orange')
        plt.title(f'Optical Flow Variance Distribution - Fold {fold_id}')
        plt.xlabel('Variance')
        plt.ylabel('Density')
        plt.legend()
        plt.savefig(f"{CONFIG['output_folder']}/optical_flow_variance_fold_{fold_id}.png", dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
    
    # Fréchet Video Distance
    fvd = calculate_fvd(train_features, test_features)
    print(f"\nFréchet Video Distance (Fold {fold_id}): {fvd:.4f}")

    # Wasserstein Distance
    wasserstein_dist = 0
    for i in range(train_features.shape[1]):
        wasserstein_dist += wasserstein_distance(train_features[:, i], test_features[:, i])
    wasserstein_dist /= train_features.shape[1]
    print(f"Wasserstein Distance (Fold {fold_id}): {wasserstein_dist:.4f}")

    return metrics

# Grid Search Helper (aligned with first code)
def build_grid(best_params):
    grid = {}
    best_n = best_params['classifier__n_neighbors']
    grid['classifier__n_neighbors'] = [max(3, best_n - 2), best_n, best_n + 2]
    best_C = best_params['classifier__C']
    grid['classifier__C'] = [best_C / 2, best_C, best_C * 2]
    best_gamma = best_params['classifier__gamma']
    if isinstance(best_gamma, str):
        grid['classifier__gamma'] = [best_gamma, 'auto']
    else:
        grid['classifier__gamma'] = [best_gamma * 0.5, best_gamma, best_gamma * 2]
    best_kernel = best_params['classifier__kernel']
    grid['classifier__kernel'] = [best_kernel, 'linear' if best_kernel != 'linear' else 'rbf']
    # Update for RobustHybridClassifier weights
    best_svm_weight = best_params['classifier__svm_weight']
    best_knn_weight = best_params['classifier__knn_weight']
    best_rf_weight = best_params['classifier__rf_weight']
    grid['classifier__svm_weight'] = [max(0.0, best_svm_weight - 0.1), best_svm_weight, min(1.0, best_svm_weight + 0.1)]
    grid['classifier__knn_weight'] = [max(0.0, best_knn_weight - 0.1), best_knn_weight, min(1.0, best_knn_weight + 0.1)]
    grid['classifier__rf_weight'] = [max(0.0, best_rf_weight - 0.1), best_rf_weight, min(1.0, best_rf_weight + 0.1)]
    return grid

# Enhanced main function with robust cross-validation (aligned with first code)
def enhanced_main():
    global train_labels  # Needed for visualization in enhanced_evaluate
    print("Loading annotations...")
    annotations = load_annotation_file(CONFIG['annotation_file'])
    
    if not annotations:
        print("No valid annotations found!")
        return
    
    video_paths = get_video_paths_with_annotations(CONFIG['input_folder'], annotations)
    labels = get_annotation_labels(video_paths, annotations)
    
    if len(video_paths) == 0:
        print("No videos found!")
        return
    
    print(f"Found {len(video_paths)} videos with {len(set(labels))} unique classes")
    print(f"Class distribution: {Counter(labels)}")
    
    label_encoder = LabelEncoder()
    encoded_labels = label_encoder.fit_transform(labels)
    train_labels = encoded_labels  # Store for visualization
    
    class_counts = Counter(encoded_labels)
    min_samples = min(class_counts.values()) if class_counts else 0
    
    if min_samples < 2:
        print("Some classes have insufficient samples for cross-validation!")
        return
    
    cv_strategy = StratifiedKFold(n_splits=min(5, min_samples), shuffle=True, random_state=CONFIG['seed'])
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    model = VideoFeatureExtractor(pretrained=True).to(device)
    
    cv_results = {
        'accuracy': [], 'balanced_accuracy': [], 'f1_weighted': [], 
        'f1_macro': [], 'precision_weighted': [], 'recall_weighted': [], 'mcc': []
    }
    
    fold_predictions = []
    fold_true_labels = []
    
    for fold_idx, (train_idx, test_idx) in enumerate(cv_strategy.split(video_paths, encoded_labels), 1):
        print(f"Fold {fold_idx} - Training samples: {len(train_idx)}, Test samples: {len(test_idx)}")
        
        train_paths = [video_paths[i] for i in train_idx]
        test_paths = [video_paths[i] for i in test_idx]
        train_labels_fold = [labels[i] for i in train_idx]
        test_labels_fold = [labels[i] for i in test_idx]
        
        train_dataset = EnhancedVideoDataset(train_paths, train_labels_fold, label_encoder, mode='train')
        test_dataset = EnhancedVideoDataset(test_paths, test_labels_fold, label_encoder, mode='test')
        
        train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False, 
                                num_workers=4, pin_memory=True)
        test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, 
                               num_workers=4, pin_memory=True)
        
        print(f"Extracting training features for fold {fold_idx}...")
        train_features, train_labels_encoded = extract_enhanced_features(model, train_loader, device)
        print(f"Extracting test features for fold {fold_idx}...")
        test_features, test_labels_encoded = extract_enhanced_features(model, test_loader, device)
        
        preprocessing_pipeline = Pipeline([
            ('scaler', RobustScaler()),
            ('feature_selection', SelectKBest(f_classif, k=min(500, train_features.shape[1]))),
            ('pca', PCA(n_components=min(100, train_features.shape[0] - 1), random_state=42))
        ])
        
        try:
            min_class_size = min(Counter(train_labels_encoded).values())
            if min_class_size >= 2 and len(set(train_labels_encoded)) > 1:
                smote = SMOTE(random_state=42, k_neighbors=min(5, min_class_size-1))
                train_features_balanced, train_labels_balanced = smote.fit_resample(
                    train_features, train_labels_encoded
                )
                print(f"Fold {fold_idx} - Applied SMOTE: {len(train_features)} -> {len(train_features_balanced)} samples")
            else:
                train_features_balanced, train_labels_balanced = train_features, train_labels_encoded
        except Exception as e:
            print(f"Fold {fold_idx} - SMOTE failed: {e}. Using original data.")
            train_features_balanced, train_labels_balanced = train_features, train_labels_encoded
        
        train_features_processed = preprocessing_pipeline.fit_transform(train_features_balanced, train_labels_balanced)
        test_features_processed = preprocessing_pipeline.transform(test_features)
        
        pipeline = Pipeline([
            ('classifier', RobustHybridClassifier())
        ])

        param_dist_random = {
            'classifier__n_neighbors': list(range(3, 21, 2)),
            'classifier__C': np.logspace(-3, 4, 8),
            'classifier__gamma': ['scale', 'auto'] + list(np.logspace(-4, 2, 7)),
            'classifier__kernel': ['rbf', 'linear', 'poly', 'sigmoid'],
            'classifier__svm_weight': np.linspace(0.0, 1.0, 11),
            'classifier__knn_weight': np.linspace(0.0, 1.0, 11),
            'classifier__rf_weight': np.linspace(0.0, 1.0, 11)
        }

        random_search = RandomizedSearchCV(
            pipeline,
            param_distributions=param_dist_random,
            n_iter=10,
            cv=5,
            scoring='f1_weighted',
            n_jobs=-1,
            verbose=1,
            random_state=42
        )
        random_search.fit(train_features_processed, train_labels_balanced)

        print(f"\nBest Random Search Params (Fold {fold_idx}):")
        for key, value in random_search.best_params_.items():
            print(f"{key}: {value}")

        param_grid = build_grid(random_search.best_params_)
        grid_search = GridSearchCV(
            pipeline,
            param_grid=param_grid,
            cv=5,
            scoring='f1_weighted',
            n_jobs=-1,
            verbose=1
        )
        grid_search.fit(train_features_processed, train_labels_balanced)

        print(f"\nBest Grid Search Params (Fold {fold_idx}):")
        for key, value in grid_search.best_params_.items():
            print(f"{key}: {value}")

        hybrid_model = grid_search.best_estimator_
        test_predictions = hybrid_model.predict(test_features_processed)
        test_probabilities = hybrid_model.predict_proba(test_features_processed)
        
        fold_metrics = enhanced_evaluate(
            test_labels_encoded, test_predictions, test_probabilities,
            label_encoder, train_features_processed, test_features_processed,
            train_dataset.temporal_features,
            test_dataset.temporal_features,
            train_dataset.flow_variances,
            test_dataset.flow_variances,
            fold_id=fold_idx
        )
        
        for metric_name, metric_value in fold_metrics.items():
            if metric_name in cv_results:
                cv_results[metric_name].append(metric_value)
        
        fold_predictions.extend(test_predictions)
        fold_true_labels.extend(test_labels_encoded)
        
        if fold_idx == 1:
            global final_train_features, final_test_features, final_train_labels, final_test_labels
            final_train_features = train_features_processed
            final_test_features = test_features_processed
            final_train_labels = train_labels_balanced
            final_test_labels = test_labels_encoded
    
    print(f"\n{'='*60}")
    print("CROSS-VALIDATION SUMMARY")
    print(f"{'='*60}")
    
    for metric_name, values in cv_results.items():
        if values:
            mean_val = np.mean(values)
            std_val = np.std(values)
            print(f"{metric_name.replace('_', ' ').title()}: {mean_val:.4f} ± {std_val:.4f}")
    
    overall_cm = confusion_matrix(fold_true_labels, fold_predictions)
    plt.figure(figsize=(12, 8))
    sns.heatmap(overall_cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_)
    plt.title('Overall Confusion Matrix (All Folds)')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(f"{CONFIG['output_folder']}/confusion_matrix_overall.png", dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    
    print(f"\n{'='*60}")
    print("OVERALL PERFORMANCE")
    print(f"{'='*60}")
    
    overall_metrics = {
        'accuracy': accuracy_score(fold_true_labels, fold_predictions),
        'balanced_accuracy': balanced_accuracy_score(fold_true_labels, fold_predictions),
        'f1_weighted': f1_score(fold_true_labels, fold_predictions, average='weighted'),
        'f1_macro': f1_score(fold_true_labels, fold_predictions, average='macro'),
        'precision_weighted': precision_score(fold_true_labels, fold_predictions, average='weighted'),
        'recall_weighted': recall_score(fold_true_labels, fold_predictions, average='weighted'),
        'mcc': matthews_corrcoef(fold_true_labels, fold_predictions),
        'kendalltau': kendalltau(fold_true_labels, fold_predictions)[0] if len(np.unique(fold_true_labels)) > 1 else 0,
        'spearmanr': spearmanr(fold_true_labels, fold_predictions)[0] if len(np.unique(fold_true_labels)) > 1 else 0
    }
    
    for name, value in overall_metrics.items():
        print(f"{name.replace('_', ' ').title()}: {value:.4f}")
    
    print(f"\n{'='*60}")
    print("CLASS-WISE PERFORMANCE ANALYSIS")
    print(f"{'='*60}")
    
    per_class_f1 = f1_score(fold_true_labels, fold_predictions, average=None)
    per_class_precision = precision_score(fold_true_labels, fold_predictions, average=None)
    per_class_recall = recall_score(fold_true_labels, fold_predictions, average=None)
    
    class_performance_df = pd.DataFrame({
        'Class': label_encoder.classes_,
        'F1_Score': per_class_f1,
        'Precision': per_class_precision,
        'Recall': per_class_recall,
        'Support': [sum(1 for x in fold_true_labels if x == i) for i in range(len(label_encoder.classes_))]
    })
    
    print(class_performance_df.round(4))
    
    results_summary = {
        'cross_validation_results': cv_results,
        'overall_metrics': overall_metrics,
        'class_wise_performance': class_performance_df.to_dict(),
        'configuration': CONFIG
    }
    
    import json
    with open(f"{CONFIG['output_folder']}/results_summary.json", 'w') as f:
        json.dump(results_summary, f, indent=2, default=str)
    
    print(f"\nResults saved to {CONFIG['output_folder']}/results_summary.json")

# Data quality checks (from first code)
def perform_data_quality_checks(video_paths, labels, annotations):
    print(f"\n{'='*60}")
    print("DATA QUALITY CHECKS")
    print(f"{'='*60}")
    
    missing_files = []
    corrupt_files = []
    
    for video_path in video_paths:
        if not os.path.exists(video_path):
            missing_files.append(video_path)
            continue
        try:
            cap = cv2.VideoCapture(video_path)
            if not cap.isOpened() or cap.get(cv2.CAP_PROP_FRAME_COUNT) == 0:
                corrupt_files.append(video_path)
            cap.release()
        except:
            corrupt_files.append(video_path)
    
    print(f"✅ Total videos: {len(video_paths)}")
    print(f"❌ Missing files: {len(missing_files)}")
    print(f"❌ Corrupt files: {len(corrupt_files)}")
    
    label_counts = Counter(labels)
    print(f"\n📊 Label Distribution:")
    for label, count in sorted(label_counts.items()):
        percentage = (count / len(labels)) * 100
        print(f"   {label}: {count} ({percentage:.1f}%)")
    
    if len(label_counts) == 0:
        print("❌ No labels found!")
        return False
        
    max_count = max(label_counts.values())
    min_count = min(label_counts.values())
    imbalance_ratio = max_count / min_count if min_count > 0 else float('inf')
    
    print(f"\n⚖️ Class Balance:")
    print(f"   Imbalance ratio: {imbalance_ratio:.2f}")
    
    if imbalance_ratio > 5:
        print("   ⚠️ Severe class imbalance detected!")
    elif imbalance_ratio > 2:
        print("   ⚠️ Moderate class imbalance detected")
    else:
        print("   ✅ Classes are relatively balanced")
    
    min_samples_cv = min(label_counts.values())
    print(f"\n🔄 Cross-Validation Feasibility:")
    print(f"   Minimum samples per class: {min_samples_cv}")
    
    if min_samples_cv < 2:
        print("   ❌ Insufficient samples for cross-validation!")
        return False
    elif min_samples_cv < 5:
        print("   ⚠️ Limited samples - consider reducing CV folds")
    else:
        print("   ✅ Sufficient samples for cross-validation")
    
    return len(missing_files) == 0 and len(corrupt_files) == 0 and min_samples_cv >= 2

# Updated main execution (aligned with first code)
if __name__ == "__main__":
    os.makedirs(CONFIG['output_folder'], exist_ok=True)
    
    print("Starting Enhanced Video Classification Pipeline")
    print(f"Configuration: {CONFIG}")
    
    # Load data for quality checks
    print("Loading annotations...")
    annotations = load_annotation_file(CONFIG['annotation_file'])
    
    if not annotations:
        print("No valid annotations found!")
    else:
        video_paths = get_video_paths_with_annotations(CONFIG['input_folder'], annotations)
        labels = get_annotation_labels(video_paths, annotations)
        
        # Perform data quality checks
        if not perform_data_quality_checks(video_paths, labels, annotations):
            print("Data quality checks failed. Exiting.")
        else:
            try:
                enhanced_main()
            except Exception as e:
                logging.error(f"Pipeline failed with error: {e}")
                import traceback
                traceback.print_exc()